# AI Finance Analyst Agent
## Phase 1 — Dataset & Data Understanding
### Notebook 01 — Dataset Metadata & Initial Profiling

This notebook performs the initial technical profiling of the `FreeFinancialTransactions50M` dataset hosted on Hugging Face.

### Objectives
- Connect to the dataset hosted on Hugging Face
- Inspect Parquet metadata
- Identify dataset dimensions
- Inspect column names and data types
- Read a small sample
- Perform initial data-quality checks
- Save initial profiling outputs

**Dataset:** `ziadatalabs/FreeFinancialTransactions50M`  
**File:** `transactions_50M.parquet`


In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
import os

print("pandas version:", pd.__version__)
print("PyArrow version:", pa.__version__)

pandas version: 3.0.6
PyArrow version: 25.0.1


## 1. Dataset Configuration

In [6]:
REPO_ID = 'ziadatalabs/FreeFinancialTransactions50M'
FILE_NAME = 'transactions_50M.parquet'

print('Repository:', REPO_ID)
print('File:', FILE_NAME)

Repository: ziadatalabs/FreeFinancialTransactions50M
File: transactions_50M.parquet


## 2. Access Dataset from Hugging Face

The dataset is accessed through Hugging Face. We do not manually download and load all 50M rows into pandas.

In [ ]:
file_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILE_NAME,
    repo_type='dataset'
)

print('Dataset local cache path:')
print(file_path)

d:\venv311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maged.magdy\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


transactions_50M.parquet: reconstructing file:   0%|          |  0.00B / 1.59GB            

transactions_50M.parquet: downloading bytes:           |  0.00B            

## 3. Open Parquet Metadata

PyArrow allows us to inspect the Parquet structure without loading the complete dataset into memory.

In [ ]:
parquet_file = pq.ParquetFile(file_path)

print('Dataset opened successfully.')

## 4. Dataset Size

The following information comes from Parquet metadata and does not require loading the complete dataset.

In [ ]:
num_rows = parquet_file.metadata.num_rows
num_columns = parquet_file.metadata.num_columns
num_row_groups = parquet_file.num_row_groups

print(f'Rows       : {num_rows:,}')
print(f'Columns    : {num_columns:,}')
print(f'Row Groups : {num_row_groups:,}')

## 5. Actual Dataset Schema

We inspect the real schema instead of assuming column names or data types.

In [ ]:
schema = parquet_file.schema_arrow

for field in schema:
    print(f'{field.name:<35} -> {field.type}')

In [ ]:
schema_df = pd.DataFrame({
    'column_name': [field.name for field in schema],
    'data_type': [str(field.type) for field in schema]
})

schema_df

## 6. Read a Small Sample

Only the first Parquet row group is read, and then the first 10,000 records are selected.

In [ ]:
table = parquet_file.read_row_group(0)
sample_df = table.to_pandas().head(10000)

print('Sample shape:', sample_df.shape)

## 7. First Look at the Data

In [ ]:
sample_df.head(10)

In [ ]:
sample_df.tail(10)

In [ ]:
sample_df.shape

## 8. Data Types

In [ ]:
sample_df.dtypes

## 9. DataFrame Information

In [ ]:
sample_df.info()

## 10. Missing Values

This is a sample-level quality check. It does not represent all 50M rows.

In [ ]:
missing_df = pd.DataFrame({
    'column': sample_df.columns,
    'missing_count': sample_df.isna().sum().values,
    'missing_pct': sample_df.isna().mean().values * 100
})

missing_df = missing_df.sort_values(
    'missing_pct',
    ascending=False
)

missing_df

## 11. Duplicate Check

Duplicate checking is performed only on the sample.

In [ ]:
duplicate_count = sample_df.duplicated().sum()

print('Duplicate rows in sample:', duplicate_count)

## 12. Descriptive Statistics

In [ ]:
sample_df.describe().T

## 13. Save Initial Profiling Outputs

In [ ]:
OUTPUT_DIR = '../outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

schema_df.to_csv(
    f'{OUTPUT_DIR}/dataset_schema.csv',
    index=False
)

missing_df.to_csv(
    f'{OUTPUT_DIR}/sample_missing_values.csv',
    index=False
)

print('Initial profiling outputs saved successfully.')

## Next Step

Review the output of this notebook before proceeding to the next profiling stage.

The next notebook will perform deeper data-quality and financial/business profiling based on the actual dataset schema.